# 12.7 Operational testing: trajectory and process health

Component and system testing ask whether the agent is right; operational testing asks whether it ran the way a governed workflow must -- in order, terminating cleanly and leaving a verifiable record. Trajectory structure reads the step record (did the five-step workflow run in the prescribed order, and on escalation did it take the path its category implies); process health counts steps, tool calls and failures and whether the run terminated within budget.

**Reference (run separately):** the operational metrics below were produced by running the scenario campaign against the system under test with the prescribed workflow order, then summarizing the rows.

```python
# Reference (run separately): heavy path that produced the numbers.
rows = evaluate.run(scns, sut, workflow_order=workflow)
print(evaluate.summary(rows))
```

**What the next cell does** -- loads the pinned operational artifact:

1. **Locate the file.** Start at `Path.cwd()` and walk up parent directories until `data/capstone_run.json` is found, so the cell runs from `notebooks/` or the repo root.
2. **Load it.** Read the file and `json.loads` it into the dict `D`, then print the resolved path.

In [ ]:
import json
from pathlib import Path

# Resolve the artifact whether we run from notebooks/ or the repo root.
root = Path.cwd().parent / 'code'
while not (root / 'data' / 'capstone_run.json').exists() and root != root.parent:
    root = root.parent
D = json.loads((root / 'data' / 'capstone_run.json').read_text())
print('loaded capstone_run.json from', root / 'data' / 'capstone_run.json')

**What the next cell does** -- reads the campaign-level operational signals:

1. **Pull the metrics.** Read `D['n_runs']` into `n_runs`, `D['workflow_adherence']` into `adherence` and `D['audit_verifies']` into `audit`.
2. **Report them.** Print the scenario count, the workflow-adherence fraction and the audit-verifies flag -- trajectory and process-health signals, not bare outcome accuracy.

In [ ]:
# Operational health across the campaign. We report trajectory and process-health
# signals, NOT the bare outcome accuracy (which would conflate a clean escalation
# with a run that terminated without an answer).
n_runs = D['n_runs']
adherence = D['workflow_adherence']
audit = D['audit_verifies']

print(f'Scenarios            : {n_runs}')
print(f'Workflow adherence   : {adherence:.3f}')
print(f'Audit verifies       : {audit}')

Workflow adherence is perfect: the plausibility gate held the five-step sequence in order on every phrasing, so no failure is the agent firing tools out of turn. The audit log verifies across the whole campaign, so any failure surfaced elsewhere is behavioral, not an integrity failure of the record. Reducing a run to one bit would conflate a clean escalation at the first gate with a workflow that terminated without an answer -- reading the trajectory and process columns keeps them distinct.

**What the next cell does** -- asserts the pinned operational metrics are intact:

1. **Keys present.** Assert `D` contains `workflow_adherence` and `audit_verifies`.
2. **Values hold.** Assert `D['workflow_adherence'] == 1.0`, `D['audit_verifies'] is True` and `D['n_runs'] == 120`, then print that the self-check passed.

In [ ]:
# Self-check on the pinned operational metrics.
assert 'workflow_adherence' in D and 'audit_verifies' in D
assert D['workflow_adherence'] == 1.0, 'workflow adherence must be perfect'
assert D['audit_verifies'] is True, 'audit log must verify'
assert D['n_runs'] == 120
print('self-check passed')